# Country Details

In [ ]:
#| default_exp game/country

In [ ]:
#| export
import sys
import math
import numpy as np
import random
from fastcore.basics import patch
import heapq # for shortest path


In [ ]:
#| export

from HexMagic.styles import   SVGBuilder
from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord 
from HexMagic.primitives import HexGrid, HexPosition ,  HexRegion , windy_edge , unique_windy_edge


In [ ]:
#| export
from HexMagic.game.kingdom import GameBoard,Kingdom,TradeRoute,Terrain,StyleCSS, Hex, TerraDemo, Geology, DrainageBasins, CountryFlag

In [ ]:
#| export
class CountryDetails:
    def __init__(self,country:Kingdom):
        self.country = country
        self.updateParent(country.world.terrain)

    def updateParent(self,parent:Terrain):
        self.parent = parent
        grid, subregion, regionMapper = self.country.region.crop_to_centered_grid(style=StyleCSS("base",fill="lightgray",stroke="blue"))
        self.subregion = subregion
        self.regionMapper = regionMapper
        self.countryMap = Terrain(bounds = grid.bounds,
            radius = grid.radius,
            colorLevels= parent.colorLevels,
            seaLevel = parent.seaLevel,
            elevationDelta = parent.elevationDelta,
            geo = parent.geo,
            climate = parent.climate
        )
        numHexes = len(grid.hexes)
        self.countryMap.hexGrid = grid
        self.countryMap.elevations = np.zeros(numHexes)

        field_names = list(parent.fields.keys())
        
        for aField in field_names:
            self.countryMap.fields[aField] = np.zeros(numHexes)

        mapper = {}
        for dest in range(numHexes):
            source = regionMapper(dest)
            self.countryMap.elevations[dest] = parent.elevations[source]
            mapper[source] = dest
            for aField in field_names:
                self.countryMap.fields[aField][dest] = parent.fields[aField][source]
        self.mapper = mapper
       


In [ ]:
#| export


@patch
def viewableRegion(self:CountryDetails,region)->HexRegion:
    """ return a region that have hexes that work """
    hexes = set([self.mapper[i] for i in region.hexes if i in self.mapper])
    return HexRegion(hexes=hexes, hexGrid = self.countryMap.hexGrid)



In [ ]:
#| export
@patch
def clearUnknowns(self:CountryDetails):
    clearC = StyleCSS("blank",fill="none",stroke="none")
    self.countryMap.hexGrid.builder.add_style(clearC)
    for i in range(len(self.countryMap.hexGrid.hexes)):
        if self.regionMapper(i) < 0:
            self.countryMap.hexGrid.hexes[i].style = clearC

In [ ]:
#| export
@patch
def unknownOverlay(self:CountryDetails,fill="white"):
    clearC = StyleCSS("unknownOverlayStyle",fill=fill,stroke="none")
    self.countryMap.hexGrid.builder.add_style(clearC)
    hexes = set([i for i in range(len(self.countryMap.hexGrid.hexes)) if self.regionMapper(i) < 0])
    clearRegion = HexRegion(hexes=hexes,hexGrid=self.countryMap.hexGrid)
    return clearRegion.draw(clearC)

In [ ]:
def califorina_place(top_n=5):

    sampleMap =  TerraDemo().bayArea_map() # TerraDemo().california_map()
    sampleMap.carve_to_ocean(num_lakes=1)
    sampleMap.hexGrid.adjustRadius(10)
    sampleWorld = GameBoard(sampleMap,top_n=6)
    sampleWorld.expand_kingdoms(max_rounds=50)
    for country in sampleWorld.kingdoms:
        neighbors = country.find_adjacent_kingdoms(sampleWorld.kingdoms)
        for dest in neighbors:
            origin = country.settlements[0]
            path = sampleWorld.find_path_dijkstra(origin, sampleWorld.kingdoms[dest].settlements[0])
            if path is not None:
                country.routes.append(TradeRoute(path,origin=origin))
        print(f"{country.countryId} {country.countryName} has {len(country.routes)} routes and {len(country.region.hexes)}")
    
    return sampleWorld

cali = califorina_place()

Done at iter 1: 1 lakes


Expansion complete after 7 rounds
1 Fred Fields has 2 routes and 73
2 Roy Range has 0 routes and 51
3 Marsh of Margaret has 0 routes and 41
4 Grace's Gap has 1 routes and 30
5 Forest of Frank has 1 routes and 93
6 Mountains of Mildred has 1 routes and 61


In [ ]:
ourCountry = cali.kingdoms[4]
print(f"{ourCountry.countryId} {ourCountry.countryName} has {len(ourCountry.routes)} routes and {len(ourCountry.region.hexes)}")

5 Forest of Frank has 1 routes and 93


In [ ]:
caliDisplay = CountryDetails(ourCountry)

In [ ]:
caliDisplay.subregion

HexRegion(hexes={47, 48, 65, 66, 67, 82, 83, 84, 85, 86, 87, 88, 89, 100, 102, 103, 104, 105, 106, 107, 108, 109, 110, 119, 120, 121, 122, 123, 124, 140, 141, 142, 160, 161, 165, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 215, 216, 217, 218, 219, 220, 221, 222, 233, 234, 235, 236, 237, 238, 239, 240, 253, 254, 255, 256, 257, 258, 273, 274, 275, 276, 277, 292, 293, 294, 295, 312, 313, 314, 330, 331, 332, 350}, hexGrid=<HexMagic.plot.hex.HexGrid object>)

In [ ]:
caliDisplay.viewableRegion(ourCountry.region)

HexRegion(hexes={47, 48, 65, 66, 67, 82, 83, 84, 85, 86, 87, 88, 89, 100, 102, 103, 104, 105, 106, 107, 108, 109, 110, 119, 120, 121, 122, 123, 124, 140, 141, 142, 160, 161, 165, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 215, 216, 217, 218, 219, 220, 221, 222, 233, 234, 235, 236, 237, 238, 239, 240, 253, 254, 255, 256, 257, 258, 273, 274, 275, 276, 277, 292, 293, 294, 295, 312, 313, 314, 330, 331, 332, 350}, hexGrid=<HexMagic.plot.hex.HexGrid object>)

In [ ]:
caliDisplay.countryMap.colorMap()
caliDisplay.clearUnknowns()
caliDisplay.countryMap.hexGrid.update()
caliDisplay.countryMap.hexGrid.builder.show()

In [ ]:
ourCountry = cali.kingdoms[2]
caliDisplay = CountryDetails(ourCountry)
caliDisplay.countryMap.colorMap()
caliDisplay.clearUnknowns()
caliDisplay.countryMap.hexGrid.update()
caliDisplay.countryMap.hexGrid.builder.show()

In [ ]:
def showWorld(sampleWorld:GameBoard,show_trade=False):
    terrain = sampleWorld.terrain
    grid = terrain.hexGrid
    builder = grid.builder
    builder.layers = []
    for country in sampleWorld.kingdoms:
        neighbors = country.find_adjacent_kingdoms(sampleWorld.kingdoms)
        for dest in neighbors:
            origin = country.settlements[0]
            path = sampleWorld.find_path_dijkstra(origin, sampleWorld.kingdoms[dest].settlements[0])
            if path is not None:
                country.routes.append(TradeRoute(path,origin=origin))

    terrain.colorMap()
    terrain.compute_climate()
    
    terrain.terrainCream()

    builder.adjust("climates", terrain.dottedClimate())
    builder.adjust("settlement",sampleWorld.settlements_overlay())
    builder.adjust("countries", sampleWorld.countries_overlay())
    builder.adjust("water", sampleWorld.world.basins.draw_watersheds())
    builder.adjust("names",sampleWorld.names_overlay())
    
    if show_trade:
        builder.adjust("trade",sampleWorld.trade_overlay())

    return builder.show()

In [ ]:
showWorld(cali)

In [ ]:
#| export
@patch
def countries_overlay(self: CountryDetails,countries:[Kingdom]) -> str:
    """Create overlay showing kingdom territories with windy borders.
    """
   
    overlay = ""
    borders = {}  # Shared border cache
    terrain = self.countryMap
    grid = terrain.hexGrid

    for country in countries:
        destRegion = self.viewableRegion(country.region)
        if len(destRegion.hexes) > 0:
            style = country.flag.kingStyle(f"{country.flag.name}_{country.countryId}")
            grid.builder.add_style(style)

            for path in destRegion.trace_perimeter_cached(
                borders,
                style=style,
                f=unique_windy_edge(iterations=2, offset_min=0.05, offset_max=0.15)
            ):
                overlay += path.svg()

    return overlay


In [ ]:
#| export
@patch
def names_overlay(self: CountryDetails,countries:[Kingdom]) -> str:
    """Create overlay showing kingdom territories with windy borders and labels."""
    terrain = self.countryMap
    grid = terrain.hexGrid
    
    # Add label style
    label_style = StyleCSS("country_label", fill="#000", stroke="none")
    grid.builder.add_style(label_style)
    
    overlay = ""
    for country in countries:
        destRegion = self.viewableRegion(country.region)
        if len(destRegion.hexes) > 0:
            # Add country name at centroid
            centroid_idx = destRegion.centroid_hex()
            if centroid_idx >= 0 and country.countryName:
                hex_obj = grid.hexes[centroid_idx]
                cx, cy = hex_obj.center.x, hex_obj.center.y
                overlay += f'\t<text x="{cx}" y="{cy}" text-anchor="middle" dominant-baseline="middle" class="country_label">{country.countryName}</text>\n'

    return overlay


In [ ]:
#| export
@patch
def settlements_overlay(self: CountryDetails, countries:[Kingdom],
                        capital_size: float = 15,
                        city_size: float = 8) -> str:
    """Create overlay showing settlements with stars for capitals and circles for cities.
    
    Uses kingdom style colors with higher saturation for capitals.
    """
    terrain = self.countryMap
    grid = terrain.hexGrid
    overlay = ""
    
    for kingdom in countries:
        # Create capital style (more saturated version of kingdom color)
        capital_style = kingdom.flag.contrastStyle(f"capital_{kingdom.countryId}")
        # Increase saturation
        capital_style.properties["fill"] = capital_style.saturate(1.5).properties["fill"]
        grid.builder.add_style(capital_style)
        
        # City style (kingdom color)
        city_style = kingdom.flag.contrastStyle(f"city_{kingdom.countryId}")
        grid.builder.add_style(city_style)
        
        for i, settlement_org in enumerate(kingdom.settlements):
            if settlement_org in self.mapper:
                settlement_idx = self.mapper[settlement_org]
                hex_obj = grid.hexes[settlement_idx]
                cx, cy = hex_obj.center.x, hex_obj.center.y
                
                if i == 0:  # Capital - draw star
                    points = []
                    for j in range(5):
                        # Outer point
                        angle = (j * 144 - 90) * math.pi / 180
                        points.append(MapCord(
                            cx + capital_size * math.cos(angle),
                            cy + capital_size * math.sin(angle)
                        ))
                        # Inner point
                        angle = (j * 144 + 72 - 90) * math.pi / 180
                        inner_r = capital_size * 0.4
                        points.append(MapCord(
                            cx + inner_r * math.cos(angle),
                            cy + inner_r * math.sin(angle)
                        ))
                    
                    star_path = MapPath(points, capital_style).closed()
                    overlay += star_path.drawClosed()
                else:  # Regular city - draw circle
                    overlay += f'\t<circle cx="{cx}" cy="{cy}" r="{city_size}" class="{city_style.name}"/>\n'
    
    return overlay

In [ ]:
def showClimes(board:GameBoard,index=0):
    if index >= len(board.kingdoms):
        index =  0
    showCountry = board.kingdoms[index]
    caliDisplay = CountryDetails(showCountry)  
    terrain = caliDisplay.countryMap
    grid = terrain.hexGrid
    grid.adjustRadius(20)
    builder = grid.builder
    builder.layers = []

   
    terrain.colorMap()
    terrain.compute_climate()
    
    terrain.terrainCream()
    

    builder.adjust("climates", terrain.dottedClimate())
    builder.adjust("settlement",caliDisplay.settlements_overlay(board.kingdoms))
    
    builder.adjust("countries",caliDisplay.countries_overlay(board.kingdoms))

    basins = DrainageBasins(terrain)
    #builder.adjust("countries", board.countries_overlay())
    builder.adjust("water", basins.draw_watersheds(max_width=4))
    builder.adjust("names",caliDisplay.names_overlay(board.kingdoms))
    builder.adjust("block",caliDisplay.unknownOverlay())

    return builder.show()

In [ ]:
showClimes(cali,4)